# Predios oficiales del evento

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad de análisis

Archivo: `data/oficiales/Datosprediales.json`. Una fila representa un registro predial del evento; puede ser lote sin edificación registrada. No representa el universo de edificios de Cali.
La fecha de descarga no equivale a fecha de verificación del objeto en terreno.

In [ ]:
PATH = ROOT / 'data/oficiales/Datosprediales.json'
meta = ficha(PATH)
raw, df, g = leer_geojson(PATH)
print("Archivo:", meta["archivo"], "| filas:", len(df), "| propiedades:", len(df.columns))
display(pd.Series(meta, name="procedencia").to_frame())

## 2. Calidad por columna y geometría

In [ ]:
calidad = mostrar_perfil(df)
geo = geometria(g)
display(pd.Series(geo, name="geometria").to_frame())
resumen = {"dataset": '01_predios_oficiales', **meta, "filas": len(df), "columnas": list(df.columns),
           "calidad_columnas": calidad.to_dict("records"), "geometria": geo}

## 3. Llaves, categorías y calidad del puntaje

In [ ]:
llaves = pd.DataFrame([{"campo": c, "nulos": int(df[c].isna().sum()),
                       "distintos_no_nulos": int(df[c].nunique()),
                       "duplicados_no_nulos": int(df.loc[df[c].notna(), c].duplicated().sum())}
                      for c in ["OBJECTID", "OBJECTID_1", "ID_PREDIO"]])
display(llaves)
estados = conteos(df["Estado_Estructural"], "estado")
estados["porcentaje"] = 100 * estados.filas / len(df)
display(estados)
assert estados.filas.sum() == len(df)
score = pd.to_numeric(df["Indice_Escombros"], errors="coerce")
res_estado = df.assign(puntaje=score).groupby("Estado_Estructural", dropna=False).puntaje.agg(
    filas="size", con_puntaje="count", minimo="min", mediana="median", maximo="max")
display(res_estado)
grafico_barras(estados[["estado", "filas"]], "Registros del evento por categoría satelital", "01_estados")
resumen.update({"llaves": llaves.to_dict("records"), "estados": estados.to_dict("records"),
                "puntaje_por_estado": res_estado.reset_index().to_dict("records")})

## 4. Lotes, pisos desconocidos y cobertura territorial

In [ ]:
pisos = pd.to_numeric(df.NPISOS, errors="coerce")
comuna = df.COMUNA.astype("string").str.strip().replace("", pd.NA)
barrio = df.BARRIO.astype("string").str.strip().replace("", pd.NA)
pisos_validos = pisos.between(1, 60)
calidad_pisos = pd.Series({"total": len(df), "nulos": int(pisos.isna().sum()),
                           "ceros": int(pisos.eq(0).sum()),
                           "fuera_1_60_incluye_lotes_no_son_errores": int((~pisos_validos & pisos.notna()).sum()),
                           "pisos_desconocidos_null": int(pisos.isna().sum()),
                           "pct_lotes_sin_edificacion_registrada": round(100*pisos.eq(0).mean(),2)})
display(calidad_pisos.to_frame("valor"))
por_comuna = pd.DataFrame({"comuna": comuna, "pisos": pisos.where(pisos_validos)})
territorio = por_comuna.groupby("comuna", dropna=False).agg(
    predios=("pisos", "size"), pisos_validos=("pisos", "count"), mediana_pisos_validos=("pisos", "median"))
territorio["pct_muestra"] = 100 * territorio.predios / len(df)
display(territorio)
reuso = pd.DataFrame({"comuna": comuna,"barrio": barrio}).dropna().groupby("barrio").comuna.nunique()
print("Códigos de barrio presentes en más de una comuna:", int((reuso>1).sum()))
grafico_barras(territorio.reset_index().fillna({"comuna":"Sin código"})[["comuna","predios"]],
                "Cobertura del inventario: predios por comuna declarada", "01_comunas")
resumen.update({"pisos": calidad_pisos.to_dict(), "comunas": territorio.reset_index().to_dict("records"),
                "comunas_distintas_no_nulas": int(comuna.nunique()),
                "codigos_barrio_reutilizados_entre_comunas": int((reuso>1).sum())})

## 5. Geometrías y diferencia entre porcentaje por número y por área de predios

In [ ]:
from shapely.validation import explain_validity
invalidas = g.geometry.notna() & ~g.geometry.is_empty & ~g.is_valid
razones = conteos(g.loc[invalidas].geometry.map(lambda x: explain_validity(x).split("[")[0]), "motivo")
display(razones)
# Copia para diagnosticar; no se escribe esta geometría como plata.
reparadas = g.geometry.make_valid()
area = gpd.GeoSeries(reparadas, crs=g.crs).to_crs(CRS_METROS).area
sin_satelite = df.Estado_Estructural.eq("Sin datos satelitales")
area_original = pd.to_numeric(df["Shape__Area"],errors="coerce")
res_area = {"n_edificios_sin_dato": int(sin_satelite.sum()),
            "pct_edificios_sin_dato": round(100*sin_satelite.mean(),2),
            "pct_huella_m2_sin_dato": round(100*area[sin_satelite].sum()/area.sum(),2),
            "pct_shape_area_original_sin_dato": round(100*area_original[sin_satelite].sum()/area_original.sum(),2),
            "area_total_huella_m2": float(area.sum()),
            "geometrias_invalidas_antes": int(invalidas.sum()),
            "geometrias_invalidas_despues_make_valid": int((~reparadas.is_valid).sum()),
            "tipos_despues_make_valid": reparadas.geom_type.value_counts().to_dict()}
display(pd.Series(res_area).to_frame("valor"))
print("El área de los polígonos prediales no equivale a área construida, habitantes u ocupación efectiva.")
print("Los conteos de otra sesión (por ejemplo, geometrías inválidas) son antecedentes; aquí manda este archivo y esta versión de GEOS.")
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(["Por número de predios", "Por área de polígonos"],
       [res_area["pct_edificios_sin_dato"], res_area["pct_huella_m2_sin_dato"]],color=["#226e86","#b87932"])
ax.set(ylabel="Porcentaje sin dato satelital",title="Dos denominadores que deben mantenerse separados")
fig.tight_layout();fig.savefig(SALIDAS/"01_sin_dato.png",dpi=150);plt.show()
resumen.update({"areas_y_geometrias": res_area,"invalidas_motivos":razones.to_dict("records")})

## 6. Contornos repetidos con identificadores distintos

Una llave única por fila no garantiza una huella física única. Se audita sin eliminar registros.

In [ ]:
wkb = g.geometry.to_wkb()
repetidos = wkb.duplicated(keep=False)
por_contorno = df.assign(_contorno=wkb).loc[repetidos].groupby("_contorno").agg(
    filas=("OBJECTID", "size"), estados_distintos=("Estado_Estructural", "nunique"),
    comunas_distintas=("COMUNA", "nunique"))
# Solo estadísticas agregadas; no se muestran los contornos ni los identificadores.
qa_repetidos = {"filas_en_grupos_contorno_repetido": int(repetidos.sum()),
                "grupos_contorno_repetido": len(por_contorno),
                "grupos_con_estados_distintos": int((por_contorno.estados_distintos>1).sum()),
                "grupos_con_comunas_distintas": int((por_contorno.comunas_distintas>1).sum()),
                "area_huella_suma_m2": float(area.sum()),
                "area_huella_union_m2": float(gpd.GeoSeries(reparadas, crs=g.crs).to_crs(CRS_METROS).union_all().area)}
display(pd.Series(qa_repetidos).to_frame("valor"))
resumen["contornos_repetidos"] = qa_repetidos
print("No se eliminan automáticamente: verificar si representan duplicación o unidades diferentes del inventario.")

## 7. Fidelidad geométrica de la serialización de bronce

Se compara la geometría original con WKT redondeado y WKT de precisión completa. La comparación permite separar errores de fuente de errores introducidos en la carga.

In [ ]:
wkt_redondeado = gpd.GeoSeries.from_wkt(g.geometry.to_wkt(), crs=g.crs)
wkt_completo = gpd.GeoSeries.from_wkt(g.geometry.to_wkt(rounding_precision=-1), crs=g.crs)
fidelidad = {"invalidas_original": int((~g.is_valid).sum()),
             "invalidas_wkt_predeterminado": int((~wkt_redondeado.is_valid).sum()),
             "invalidas_wkt_precision_completa": int((~wkt_completo.is_valid).sum()),
             "validas_que_redondeo_hace_invalidas": int((g.is_valid & ~wkt_redondeado.is_valid).sum()),
             "filas_geometria_exacta_precision_completa": int(g.geometry.geom_equals_exact(wkt_completo,0).sum())}
display(pd.Series(fidelidad).to_frame("resultado"))
assert fidelidad["filas_geometria_exacta_precision_completa"] == len(g)
resumen["fidelidad_serializacion"] = fidelidad
print("Decisión: serializar WKT con rounding_precision=-1 en bronze/silver; conservar los bytes GeoJSON originales.")

## 8. Semántica catastral confirmada: cero pisos representa lote

Aclaración del dueño durante esta revisión: `NPISOS=0` significa predio sin edificación registrada.
Se conserva como lote y se excluye de promedios de altura; no es un fallo de carga. Los valores
NULL siguen siendo desconocidos. El cruce con EDIFPRED es una comprobación de coherencia,
no un reemplazo de esta definición. Catastro y señal satelital pueden describir fechas y unidades diferentes.

In [ ]:
lote=pisos.eq(0)
edif=df.EDIFPRED.astype("string").str.strip().replace("",pd.NA)
estado_predio=pd.Series(np.select([lote,pisos.isna()],["Sin edificación registrada","Pisos desconocidos"],default="Con pisos registrados"))
semantica={"predios_lote":int(lote.sum()),"pct_lotes":round(100*lote.mean(),2),
           "pisos_null":int(pisos.isna().sum()),"predios_con_pisos_positivos":int(pisos.gt(0).sum()),
           "lotes_edifpred_nulo_o_blanco":int((lote & edif.isna()).sum()),
           "pct_lotes_edifpred_nulo_o_blanco":round(100*edif[lote].isna().mean(),2),
           "lotes_edifpred_codigo_00":int((lote & edif.eq("00").fillna(False)).sum()),
           "lotes_edifpred_codigo_01":int((lote & edif.eq("01").fillna(False)).sum()),
           "promedio_pisos_solo_positivos":float(pisos[pisos.gt(0)].mean()),
           "mediana_pisos_solo_positivos":float(pisos[pisos.gt(0)].median())}
display(pd.Series(semantica).to_frame("valor"))
cruce=pd.crosstab(estado_predio,df.Estado_Estructural)
display(cruce)
resumen["semantica_catastral"]=semantica
resumen["estado_predio_vs_satelite"]=cruce.reset_index().to_dict("records")
print("Conservar posibles discrepancias catastro/satélite; no inventar un edificio sobre el lote ni borrar su señal satelital.")

## Reglas decididas para plata

- **Incorporar como base principal.** Usar `OBJECTID` como llave dentro de esta versión del archivo; no `ID_PREDIO`.
- Conservar los códigos originales como texto, recortar espacios en columnas nuevas y usar `(comuna, barrio)` como llave territorial compuesta.
- Conservar puntaje original y categoría; `Indice_Escombros` tiene signo y no es una probabilidad 0–1. «Sin cambio» y «sin datos» no equivalen a «sin daño».
- `NPISOS=0` significa lote sin edificación registrada, según aclaración del dueño: conservar cero y `sin_edificacion_registrada=True`; excluirlo del promedio de pisos y del ranking de inspección de edificios. No imputarlo con mediana ni GEM. Los 11 nulos son desconocidos, una categoría distinta. Si se estima alguno de esos nulos, registrar evidencia, método y bandera.
- Reparar geometrías únicamente en plata, preservando original, motivo y cambio de tipo/área. No borrar sin un registro de cuarentena.
- Reportar tasas sobre la muestra cubierta, con conteos y huellas como denominadores distintos. No extrapolar resultados de las comunas presentes al total de Cali.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('01_predios_oficiales', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")